In [51]:
import torch 
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

In [52]:
# read it in to inspect it
from matplotlib.pylab import test


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(len(text))
print(text[:100])
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [53]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [54]:
n = int(.9*len(text))
data = torch.tensor(encode(text), dtype=torch.long)
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [55]:
torch.manual_seed(1337) 
block_size = 8
batch_size = 4
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0,len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [56]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # each index maps to a vector of logits for the next token
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # becomes (B,C)
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1) # append to the running sequence (B,T+1)
        return idx
m = BigramLanguageModel(vocab_size)
logits,loss = m(xb,yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [57]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(10000):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

2.362440586090088

M:
IUSh t,
F th he d ke alved.
Thupld, cipbll t
I: ir w, l me sie hend lor ito'l an e

I:
Gochosen e


In [58]:
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)


xbow = torch.zeros((B,T,C), dtype=torch.float)
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)

print(xbow)

torch.Size([4, 8, 2])
tensor([[[ 0.4291, -1.0265],
         [-0.1932, -0.9308],
         [ 0.0489, -1.8460],
         [ 0.2217, -1.5188],
         [ 0.2534, -1.0912],
         [ 0.1843, -0.9350],
         [ 0.1406, -0.8636],
         [ 0.0451, -0.5761]],

        [[-1.1361, -0.3311],
         [ 0.0865,  0.1226],
         [-0.0399, -0.3780],
         [-0.2481, -0.6573],
         [-0.4083, -0.6666],
         [-0.4022, -0.2765],
         [-0.2366, -0.1957],
         [-0.1285, -0.3677]],

        [[ 0.0294, -0.3281],
         [ 0.1667, -0.1321],
         [ 0.0364,  0.0627],
         [-0.0555, -0.5750],
         [-0.0919, -0.4889],
         [ 0.3176, -0.4780],
         [ 0.1882, -0.5250],
         [ 0.2641, -0.2078]],

        [[ 0.3954,  1.2923],
         [ 0.0169,  0.3606],
         [-0.1399,  0.5667],
         [-0.1251,  0.5800],
         [-0.4911,  0.7454],
         [-0.3138,  0.5580],
         [-0.3148,  0.7044],
         [-0.5545,  0.6191]]])


In [59]:
# attention weights concept


trill = torch.tril(torch.ones(T,T))
print(trill)
wei = torch.zeros(T,T)
wei = wei.masked_fill(trill == 0, float('-inf'))
print(wei)

wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(xbow3)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[[ 0.4291, -1.0265],
         [-0.1932, -0.9308],
         [ 0.0489, -1.8460],
         [ 0.2217, -1.5188],
         [ 0.2534, -1.0912],
         [ 0.1843, -0.9350],
         [ 0.1406, -0.8636],
         [ 0.0451, -0.5761]],

        [[-1.1361, -0.3311],
       

In [ ]:
# self attention

torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x)
wei = q @ k.transpose(-2, -1)


trill = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(trill == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ v
print(out.shape)

tensor([[[-1.5713e-01,  8.8009e-01,  1.6152e-01, -7.8239e-01, -1.4289e-01,
           7.4676e-01,  1.0068e-01, -5.2395e-01, -8.8726e-01,  1.9068e-01,
           1.7616e-01, -5.9426e-01, -4.8124e-01, -4.8598e-01,  2.8623e-01,
           5.7099e-01],
         [ 3.3749e-01,  3.2856e-02, -8.1365e-02, -1.3163e-01, -1.3405e-01,
           1.2847e-01, -2.1399e-01, -2.7767e-01, -3.9002e-01,  5.0880e-01,
           4.9469e-01, -3.1069e-01, -1.8522e-03,  6.0111e-03,  7.4269e-02,
           9.8672e-01],
         [ 4.2615e-01, -6.1414e-02, -2.5954e-01,  4.7843e-02,  2.1579e-02,
          -4.4680e-02, -9.8932e-02, -1.1272e-01, -3.0332e-01,  4.4928e-02,
           3.8727e-01, -5.2057e-02, -1.4360e-01, -2.1577e-02, -4.5567e-02,
           1.1203e+00],
         [ 4.8603e-01, -2.2346e-01, -3.4712e-01,  1.4459e-01,  2.4057e-01,
          -2.6597e-01, -5.7118e-02, -3.7961e-02, -9.2482e-02,  9.4373e-02,
           1.1739e-01,  8.5392e-02,  1.3556e-02,  1.4901e-01,  1.8500e-01,
           8.5887e-01],
    